In [29]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB 640.0 kB/s eta 0:02:39
   ---------------------------------------- 0.1/101.7 MB 1.8 MB/s eta 0:00:57
   ---------------------------------------- 0.3/101.7 MB 2.0 MB/s eta 0:00:52
   ---------------------------------------- 0.3/101.7 MB 1.8 MB/s eta 0:00:57
   ---------------------------------------- 0.5/101.7 MB 2.2 MB/s eta 0:00:46
   ---------------------------------------- 0.6/101.7 MB 2.3 MB/s eta 0:00:45
   ---------------------------------------- 0.7/101.7 MB 2.3 MB/s eta 0:00:44
   ---------------------------------------- 0.7/101.7 MB 2.3 MB/s eta 0:00:45
   ---------------------------------------- 0.7/101.7 MB 2.3 MB/s eta 0:00:45
   ---------------------------------------- 0.9/101.7 MB 2.2 MB/s eta 0:00:46
   ---------------------------------------- 1.0/101.7 MB 2.2 MB/s eta 0:00:46
   ---------------------------------------- 1.1/101.7 MB 2.2 MB/s eta


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install optuna

   ---------------------------------------- 0.0/419.5 kB ? eta -:--:--
    --------------------------------------- 10.2/419.5 kB ? eta -:--:--
   -- ------------------------------------ 30.7/419.5 kB 640.0 kB/s eta 0:00:01
   ----- --------------------------------- 61.4/419.5 kB 656.4 kB/s eta 0:00:01
   -------- ------------------------------ 92.2/419.5 kB 655.4 kB/s eta 0:00:01
   ----------- -------------------------- 122.9/419.5 kB 654.9 kB/s eta 0:00:01
   --------------- ---------------------- 174.1/419.5 kB 697.2 kB/s eta 0:00:01
   ------------------ ------------------- 204.8/419.5 kB 689.9 kB/s eta 0:00:01
   ---------------------- --------------- 245.8/419.5 kB 752.5 kB/s eta 0:00:01
   ------------------------- ------------ 286.7/419.5 kB 768.0 kB/s eta 0:00:01
   ----------------------------- -------- 327.7/419.5 kB 780.0 kB/s eta 0:00:01
   --------------------------------- ---- 368.6/419.5 kB 791.2 kB/s eta 0:00:01
   -------------------------------------- 419.5/419.5 kB 


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor , RandomForestRegressor , GradientBoostingRegressor , AdaBoostRegressor
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 104,4.45,4,5,3+,Relatively New,7222.0,0,0,1,Medium,High Floor
1,flat,sector 86,1.26,3,3,3,Moderately Old,1747.0,1,0,1,Medium,Mid Floor
2,flat,sector 9a,1.50,4,3,3+,Relatively New,1833.0,0,0,0,Low,Mid Floor
3,flat,sector 37d,0.75,2,2,3,Moderately Old,1410.0,0,0,1,Medium,High Floor
4,flat,sector 86,1.45,3,2,3,Relatively New,1480.0,0,0,2,Medium,Mid Floor


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3554 non-null   object 
 1   sector           3554 non-null   object 
 2   price            3554 non-null   float64
 3   bedRoom          3554 non-null   int64  
 4   bathroom         3554 non-null   int64  
 5   balcony          3554 non-null   object 
 6   agePossession    3554 non-null   object 
 7   built_up_area    3554 non-null   float64
 8   servant room     3554 non-null   int64  
 9   store room       3554 non-null   int64  
 10  furnishing_type  3554 non-null   int64  
 11  luxury_category  3554 non-null   object 
 12  floor_category   3554 non-null   object 
dtypes: float64(2), int64(5), object(6)
memory usage: 361.1+ KB


In [5]:
df['furnishing_type'].value_counts()

furnishing_type
1    2374
0     995
2     185
Name: count, dtype: int64

In [6]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [7]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 104,4.45,4,5,3+,Relatively New,7222.0,0,0,semifurnished,Medium,High Floor
1,flat,sector 86,1.26,3,3,3,Moderately Old,1747.0,1,0,semifurnished,Medium,Mid Floor
2,flat,sector 9a,1.50,4,3,3+,Relatively New,1833.0,0,0,unfurnished,Low,Mid Floor
3,flat,sector 37d,0.75,2,2,3,Moderately Old,1410.0,0,0,semifurnished,Medium,High Floor
4,flat,sector 86,1.45,3,2,3,Relatively New,1480.0,0,0,furnished,Medium,Mid Floor


In [8]:
X = df.drop(columns=['price'])
y = df['price']

In [9]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [10]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3554 non-null   object 
 1   sector           3554 non-null   object 
 2   bedRoom          3554 non-null   int64  
 3   bathroom         3554 non-null   int64  
 4   balcony          3554 non-null   object 
 5   agePossession    3554 non-null   object 
 6   built_up_area    3554 non-null   float64
 7   servant room     3554 non-null   int64  
 8   store room       3554 non-null   int64  
 9   furnishing_type  3554 non-null   object 
 10  luxury_category  3554 non-null   object 
 11  floor_category   3554 non-null   object 
dtypes: float64(1), int64(4), object(7)
memory usage: 333.3+ KB


In [11]:
y_transformed.info()

<class 'pandas.core.series.Series'>
RangeIndex: 3554 entries, 0 to 3553
Series name: price
Non-Null Count  Dtype  
--------------  -----  
3554 non-null   float64
dtypes: float64(1)
memory usage: 27.9 KB


#### Ordinal Encoding

In [11]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [18]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown = 'use_encoded_value' , unknown_value = -1), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [19]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [20]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [21]:
scores.mean(),scores.std()

(np.float64(0.7320609366629445), np.float64(0.021993630707063144))

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [23]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [24]:
y_pred = pipeline.predict(X_test)

In [25]:
y_pred = np.expm1(y_pred)

In [26]:
mean_absolute_error(np.expm1(y_test),y_pred)

1.0093990721894615

In [27]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [31]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [32]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

D:\major project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [33]:
model_output

[['linear_reg', np.float64(0.7320609366629445), 1.0093990721894615],
 ['svr', np.float64(0.7567525130519119), 0.9044869776326594],
 ['ridge', np.float64(0.7320660620337753), 1.0093617946345688],
 ['LASSO', np.float64(0.05511514349379145), 1.57142740604153],
 ['decision tree', np.float64(0.7796419522067433), 0.7388650327416957],
 ['random forest', np.float64(0.8839745673800026), 0.5303683161966894],
 ['extra trees', np.float64(0.8698513994590495), 0.5949514967281752],
 ['gradient boosting', np.float64(0.8764586919336897), 0.550767330438277],
 ['adaboost', np.float64(0.7561635672013807), 0.8085947456403956],
 ['mlp', np.float64(0.8030319305125768), 0.7856235246954544],
 ['xgboost', np.float64(0.890589425747194), 0.4931351597366118]]

In [34]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [35]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.890589,0.493135
5,random forest,0.883975,0.530368
7,gradient boosting,0.876459,0.550767
6,extra trees,0.869851,0.594951
4,decision tree,0.779642,0.738865
9,mlp,0.803032,0.785624
8,adaboost,0.756164,0.808595
1,svr,0.756753,0.904487
2,ridge,0.732066,1.009362
0,linear_reg,0.732061,1.009399


#### OneHotEncoding

In [39]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown = 'use_encoded_value' , unknown_value = -1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first' , handle_unknown = 'ignore'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [40]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [41]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [42]:
scores.mean()

np.float64(0.8565889711455126)

In [43]:
scores.std()

np.float64(0.018258395020494946)

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [45]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [46]:
y_pred = pipeline.predict(X_test)

D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [47]:
y_pred = np.expm1(y_pred)

In [48]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.7342325774807559

In [49]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [50]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [51]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\skle

In [52]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [53]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.896032,0.495366
6,extra trees,0.892947,0.503654
5,random forest,0.892043,0.507155
7,gradient boosting,0.875782,0.555805
9,mlp,0.870301,0.612154
4,decision tree,0.810495,0.672115
0,linear_reg,0.856589,0.734233
2,ridge,0.856608,0.742094
8,adaboost,0.754356,0.820018
1,svr,0.760516,0.894083


#### OneHotEncoding With PCA

In [57]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown = 'use_encoded_value',unknown_value = -1), columns_to_encode),
        ('cat1',OneHotEncoder(handle_unknown = 'ignore',drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [58]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [59]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [60]:
scores.mean()

np.float64(0.05772536263146851)

In [61]:
scores.std()

np.float64(0.019020429852698237)

In [62]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [63]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [64]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\major project\venv\Lib\site-packages\skle

In [65]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [66]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.752183,0.679084
6,extra trees,0.731165,0.702769
4,decision tree,0.683399,0.750496
10,xgboost,0.608886,0.952780
7,gradient boosting,0.611192,1.043480
8,adaboost,0.284733,1.390838
1,svr,0.225244,1.392605
9,mlp,0.220301,1.505760
3,LASSO,0.055310,1.571317
2,ridge,0.057725,1.572819


#### Target Encoder

In [15]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown = 'use_encoded_value' , unknown_value = -1), columns_to_encode),
        ('cat1',OneHotEncoder(handle_unknown = 'ignore',drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [68]:
!pip install category_encoders

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/85.9 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/85.9 kB ? eta -:--:--
   ---------------------------------------- 85.9/85.9 kB 1.2 MB/s eta 0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Attempting uninstall: joblib
    Found existing installation: joblib 1.1.1
    Uninstalling joblib-1.1.1:
      Successfully uninstalled joblib-1.1.1



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [71]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [72]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [73]:
scores.mean(),scores.std()

(np.float64(0.824765261198414), np.float64(0.018795160744213172))

In [36]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [37]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [38]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [39]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [40]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.886238,0.513423
10,xgboost,0.893575,0.515358
5,random forest,0.875210,0.555254
1,svr,0.883839,0.555621
9,mlp,0.881173,0.570407
7,gradient boosting,0.859080,0.615892
4,decision tree,0.779230,0.709625
0,linear_reg,0.856684,0.729937
2,ridge,0.856854,0.739295
8,adaboost,0.721072,0.892494


#### Hyperparameter Tuning

In [12]:
from sklearn.model_selection import GridSearchCV

In [30]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Cleaned up the param_grid (Removed None from max_features)
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples': [0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['sqrt', 'log2'] # Much faster!
}

# 2. Dropped to 5 folds
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Use RandomizedSearchCV
search = RandomizedSearchCV(
    pipeline, 
    param_distributions=param_grid, 
    n_iter=20,          # ONLY test 20 random combinations instead of all of them!
    cv=kfold, 
    scoring='r2', 
    n_jobs=2, 
    verbose=4,
    random_state=42
)

# Start training!
print("Starting fast Randomized Search...")
search.fit(X, y_transformed)

print("Best R2 Score:", search.best_score_)
print("Best Params:", search.best_params_)

Starting fast Randomized Search...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best R2 Score: 0.8713140671999853
Best Params: {'regressor__n_estimators': 300, 'regressor__max_samples': 1.0, 'regressor__max_features': 'log2', 'regressor__max_depth': None}


In [29]:
columns_to_encode = ['property_type', 'balcony', 'furnishing_type', 'luxury_category', 'floor_category']

param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': [None, 'sqrt']
}
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown = 'use_encoded_value' , unknown_value = -1), columns_to_encode),
        ('cat1',OneHotEncoder(handle_unknown = 'ignore',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs= 2, verbose=4)
search.fit(X, y_transformed)


Fitting 10 folds for each of 128 candidates, totalling 1280 fits


KeyboardInterrupt: 

In [24]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': [None, 'sqrt']
}

In [17]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown = 'use_encoded_value' , unknown_value = -1), columns_to_encode),
        ('cat1',OneHotEncoder(handle_unknown = 'ignore',drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [18]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [25]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [27]:
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs= -1, verbose=4)

In [28]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits


KeyboardInterrupt: 

In [101]:
final_pipe = search.best_estimator_

In [102]:
search.best_params_

{'regressor__max_depth': None,
 'regressor__max_features': None,
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 200}

In [103]:
search.best_score_

np.float64(0.9050566425435299)

In [104]:
final_pipe.fit(X,y_transformed)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

#### Exporting the model

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown = 'use_encoded_value' , unknown_value = -1), columns_to_encode),
        ('cat1',OneHotEncoder(handle_unknown = 'ignore',drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [23]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=500))
])

In [108]:
pipeline.fit(X,y_transformed)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [109]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [110]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [111]:
X

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 104,4,5,3+,Relatively New,7222.0,0,0,semifurnished,Medium,High Floor
1,flat,sector 86,3,3,3,Moderately Old,1747.0,1,0,semifurnished,Medium,Mid Floor
2,flat,sector 9a,4,3,3+,Relatively New,1833.0,0,0,unfurnished,Low,Mid Floor
3,flat,sector 37d,2,2,3,Moderately Old,1410.0,0,0,semifurnished,Medium,High Floor
4,flat,sector 86,3,2,3,Relatively New,1480.0,0,0,furnished,Medium,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...,...
3549,flat,sector 104,3,4,3+,Relatively New,2000.0,1,0,unfurnished,Medium,Mid Floor
3550,flat,sector 69,3,3,1,Relatively New,1392.0,0,0,unfurnished,High,High Floor
3551,flat,sector 33,2,2,2,Relatively New,724.0,0,0,furnished,Low,Low Floor
3552,house,sector 104,3,5,3+,Moderately Old,920.0,0,0,semifurnished,Low,Low Floor


#### Trying out the predictions

In [112]:
X.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')

In [113]:
X.iloc[0].values

array(['flat', 'sector 104', np.int64(4), np.int64(5), '3+',
       'Relatively New', np.float64(7222.0), np.int64(0), np.int64(0),
       'semifurnished', 'Medium', 'High Floor'], dtype=object)

In [114]:
data = [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 'unfurnished', 'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']

In [115]:
# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

In [116]:
one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 102,4,3,3+,New Property,2750,0,0,unfurnished,Low,Low Floor


In [117]:
np.expm1(pipeline.predict(one_df))

array([2.83488038])

In [118]:
X.dtypes

property_type       object
sector              object
bedRoom              int64
bathroom             int64
balcony             object
agePossession       object
built_up_area      float64
servant room         int64
store room           int64
furnishing_type     object
luxury_category     object
floor_category      object
dtype: object

In [119]:
sorted(X['sector'].unique().tolist())

['dwarka expressway',
 'gwal pahari',
 'manesar',
 'sector 1',
 'sector 102',
 'sector 103',
 'sector 104',
 'sector 105',
 'sector 106',
 'sector 107',
 'sector 108',
 'sector 109',
 'sector 10a',
 'sector 11',
 'sector 110',
 'sector 111',
 'sector 112',
 'sector 113',
 'sector 12',
 'sector 13',
 'sector 14',
 'sector 15',
 'sector 17',
 'sector 17a',
 'sector 17b',
 'sector 2',
 'sector 21',
 'sector 22',
 'sector 23',
 'sector 24',
 'sector 25',
 'sector 26',
 'sector 27',
 'sector 28',
 'sector 3',
 'sector 3 phase 2',
 'sector 3 phase 3 extension',
 'sector 30',
 'sector 31',
 'sector 33',
 'sector 36',
 'sector 36a',
 'sector 37',
 'sector 37c',
 'sector 37d',
 'sector 38',
 'sector 39',
 'sector 4',
 'sector 40',
 'sector 41',
 'sector 43',
 'sector 45',
 'sector 46',
 'sector 47',
 'sector 48',
 'sector 49',
 'sector 5',
 'sector 50',
 'sector 51',
 'sector 52',
 'sector 53',
 'sector 54',
 'sector 55',
 'sector 56',
 'sector 57',
 'sector 58',
 'sector 59',
 'sector 6',
 'se

In [35]:
# 1. Count how many times each sector appears
sector_frequencies = X['sector'].value_counts()

# 2. Map those counts back into the column, replacing the text with the number
X['sector'] = X['sector'].map(sector_frequencies)

# (Optional) Check the result to see the new numbers!
print(X['sector'].head())

0    67
1    62
2    10
3    63
4    62
Name: sector, dtype: int64


In [47]:
X.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 104,4,5,3+,Relatively New,7222.0,0,0,semifurnished,Medium,High Floor
1,flat,sector 86,3,3,3,Moderately Old,1747.0,1,0,semifurnished,Medium,Mid Floor
2,flat,sector 9a,4,3,3+,Relatively New,1833.0,0,0,unfurnished,Low,Mid Floor
3,flat,sector 37d,2,2,3,Moderately Old,1410.0,0,0,semifurnished,Medium,High Floor
4,flat,sector 86,3,2,3,Relatively New,1480.0,0,0,furnished,Medium,Mid Floor


In [48]:
y_transformed.head()

0    1.695616
1    0.815365
2    0.916291
3    0.559616
4    0.896088
Name: price, dtype: float64

In [49]:
# 1. Move 'sector' here so it gets Standard Scaled alongside area and bedrooms
num_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']

# 2. Remove 'sector' from this list!
ordinal_cols = ['property_type', 'balcony', 'furnishing_type', 'luxury_category', 'floor_category']

ohe_cols = ['agePossession' , 'sector']

# 3. Preprocessor remains clean and fast
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat_ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_cols)
    ], 
    remainder='passthrough'
)

def objective(trial):
    param = {
        # 1. More estimators to allow gradual learning
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000, step=100),
        
        # 2. INCREASED DEPTH: crucial for Ordinal Encoded categories
        'max_depth': trial.suggest_int('max_depth', 6, 15), 
        
        # 3. Learning rate tuning
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        
        # 4. Gamma acts as a "pruner". Since trees are deeper, this prevents overfitting
        'gamma': trial.suggest_float('gamma', 0.01, 2.0, log=True),
        
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),
    }

    xgb_model = XGBRegressor(**param, random_state=42, n_jobs=1)
    
    current_pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', xgb_model)
    ])

    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(current_pipe, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    
    return scores.mean()

# Run the Study
study = optuna.create_study(direction='maximize')
print("Starting High-Depth Optuna optimization...")
study.optimize(objective, n_trials=50)

print("Best R2 Score:", study.best_value)
print("Best Parameters:", study.best_params)

[I 2026-04-04 16:15:50,865] A new study created in memory with name: no-name-79cd44e1-9909-4790-a4cd-e9504e52a117


Starting High-Depth Optuna optimization...


[I 2026-04-04 16:15:54,712] Trial 0 finished with value: 0.8977694137253145 and parameters: {'n_estimators': 600, 'max_depth': 12, 'learning_rate': 0.028234091099243065, 'gamma': 0.018652080279769206, 'subsample': 0.8650179900370667, 'colsample_bytree': 0.6775898250804117, 'min_child_weight': 1}. Best is trial 0 with value: 0.8977694137253145.
[I 2026-04-04 16:15:56,311] Trial 1 finished with value: 0.8324825504329697 and parameters: {'n_estimators': 500, 'max_depth': 15, 'learning_rate': 0.10738326665306377, 'gamma': 1.5143100483532912, 'subsample': 0.6979095929944914, 'colsample_bytree': 0.8825428563815253, 'min_child_weight': 4}. Best is trial 0 with value: 0.8977694137253145.
[I 2026-04-04 16:16:00,944] Trial 2 finished with value: 0.8829487379071856 and parameters: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.01267234378424308, 'gamma': 0.029637878022292417, 'subsample': 0.8327320913973497, 'colsample_bytree': 0.9825551757974972, 'min_child_weight': 5}. Best is trial 

Best R2 Score: 0.9017940212230009
Best Parameters: {'n_estimators': 700, 'max_depth': 7, 'learning_rate': 0.07491503281933405, 'gamma': 0.02700582786784373, 'subsample': 0.6683529295324979, 'colsample_bytree': 0.6346081170225597, 'min_child_weight': 1}


In [31]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)

In [32]:
y_pred = np.expm1(y_pred)

In [33]:
y_pred[0]

np.float64(1.470957212774191)

In [34]:
X.head(1)

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 104,4,5,3+,Relatively New,7222.0,0,0,semifurnished,Medium,High Floor


In [35]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.5548470046022729

In [56]:
!pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB 435.7 kB/s eta 0:03:50
   ---------------------------------------- 0.2/100.2 MB 1.7 MB/s eta 0:00:58
   ---------------------------------------- 0.6/100.2 MB 3.9 MB/s eta 0:00:26
   ---------------------------------------- 1.1/100.2 MB 5.6 MB/s eta 0:00:18
    --------------------------------------- 1.7/100.2 MB 7.9 MB/s eta 0:00:13
    --------------------------------------- 2.3/100.2 MB 8.7 MB/s eta 0:00:12
   - -------------------------------------- 3.1/100.2 MB 9.8 MB/s eta 0:00:10
   - -------------------------------------- 3.7/100.2 MB 10.2 MB/s eta 0:00:10
   - -------------------------------------- 4.8/100.2 MB 11.8 MB/s eta 0:00:09
   - -------------------------------------- 4.8/100.2 MB 11.9 MB/s eta 0:00:09
   - -------------------------------------- 4.8/100.2 MB 11.9 MB/s eta 0:00:09
   - -------------------------------------- 4.9/100.2 MB 9.5 MB/s


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
from catboost import CatBoostRegressor

In [58]:
# 1. Define your categorical columns (Notice we use the raw strings!)
cat_features = ['property_type', 'sector', 'balcony', 'agePossession', 
                'furnishing_type', 'luxury_category', 'floor_category']

# 2. Initialize CatBoost (It doesn't need a massive pipeline)
# These base parameters routinely hit 90%+ on real estate data
cat_model = CatBoostRegressor(
    iterations=1000, 
    learning_rate=0.05, 
    depth=8, 
    eval_metric='MAE',
    cat_features=cat_features,
    random_seed=42,
    verbose=100 # Prints progress every 100 trees
)

# 3. Train directly on your train split
cat_model.fit(X_train, y_train)

# 4. Predict and evaluate
y_pred_log = cat_model.predict(X_test)

# Convert back to real currency (Crores)
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)



0:	learn: 0.4103333	total: 197ms	remaining: 3m 17s
100:	learn: 0.1324972	total: 10.6s	remaining: 1m 34s
200:	learn: 0.1157398	total: 21.6s	remaining: 1m 26s
300:	learn: 0.1041064	total: 32s	remaining: 1m 14s
400:	learn: 0.0950806	total: 43.1s	remaining: 1m 4s
500:	learn: 0.0867570	total: 54.3s	remaining: 54.1s
600:	learn: 0.0792323	total: 1m 5s	remaining: 43.3s
700:	learn: 0.0727968	total: 1m 16s	remaining: 32.4s
800:	learn: 0.0682209	total: 1m 27s	remaining: 21.6s
900:	learn: 0.0641926	total: 1m 37s	remaining: 10.7s
999:	learn: 0.0599982	total: 1m 47s	remaining: 0us


NameError: name 'r2_score' is not defined

In [59]:
print("R2 Score:", r2_score(y_test_real, y_pred_real))
print("MAE: ₹", np.round(mean_absolute_error(y_test_real, y_pred_real) * 100, 2), "Lakhs")

NameError: name 'r2_score' is not defined